### 1.ChatMessageHistory的使用(存储消息)-->常用

In [21]:
#记忆的存储
from langchain_community.chat_message_histories import ChatMessageHistory

#1. 创建一个ChatMessageHistory对象

history = ChatMessageHistory()

#2.添加相关的消息进行存储

history.add_user_message("你好")

history.add_ai_message("很高兴认识你")

#3.打印存储的信息
print(history.messages)

[HumanMessage(content='你好', additional_kwargs={}, response_metadata={}), AIMessage(content='很高兴认识你', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


In [22]:
#与大模型相结合
#1.获取大模型
import os
import dotenv
from langchain_community.chat_models.tongyi import ChatTongyi
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
dotenv.load_dotenv()

chat_model =ChatTongyi(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    model_name="qwen-flash",
)

In [23]:
from langchain_community.chat_message_histories import ChatMessageHistory

#1. 创建一个ChatMessageHistory对象

history = ChatMessageHistory()

#2.添加相关的消息进行存储
history.add_user_message("你好")
history.add_ai_message("很高兴认识你")
history.add_user_message("帮我计算1+2*3=?")

response = chat_model.invoke(history.messages)
print(response.content)


我们来一步一步计算这个表达式：

表达式是：  
1 + 2 * 3

根据数学中的**运算顺序规则（先乘除后加减）**，我们要先算乘法部分。

第一步：计算 2 * 3  
2 * 3 = 6

第二步：将结果加到 1 上  
1 + 6 = 7

所以，最终答案是：  
**7** ✅


### 2.ConversationBufferMemory的使用(存储对话)-已弃用

In [ ]:
#以字符串的方式返回存储的信息
from langchain_classic.memory import ConversationBufferMemory


#1.创建一个ConversationBufferMemory对象
memory = ConversationBufferMemory()

#2.添加相关的消息进行存储
memory.save_context(inputs={"input":"你好，我叫小明"}, outputs={"output":"很高兴认识你"})

memory.save_context(inputs={"input":"1+2*3=?"}, outputs={"output":"7"})

#3.打印存储的信息
print(memory.load_memory_variables({}))
#返回的字典结构的key叫history

{'history': 'Human: 你好，我叫小明\nAI: 很高兴认识你\nHuman: 1+2*3=?\nAI: 7'}


In [25]:
#以消息列表的方式返回存储的信息
from langchain_classic.memory import ConversationBufferMemory


#1.创建一个ConversationBufferMemory对象
memory = ConversationBufferMemory(return_messages=True)

#2.添加相关的消息进行存储
memory.save_context(inputs={"input":"你好，我叫小明"}, outputs={"output":"很高兴认识你"})

memory.save_context(inputs={"input":"1+2*3=?"}, outputs={"output":"7"})

#3.打印存储的信息
#返回消息列表的方式1
print(memory.load_memory_variables({}))

#返回消息列表的方式2
print(memory.chat_memory.messages)

{'history': [HumanMessage(content='你好，我叫小明', additional_kwargs={}, response_metadata={}), AIMessage(content='很高兴认识你', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='1+2*3=?', additional_kwargs={}, response_metadata={}), AIMessage(content='7', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}
[HumanMessage(content='你好，我叫小明', additional_kwargs={}, response_metadata={}), AIMessage(content='很高兴认识你', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='1+2*3=?', additional_kwargs={}, response_metadata={}), AIMessage(content='7', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


In [ ]:
#例3结合llm,prompttemplate的使用
#1.获取大模型
import os
import dotenv
from langchain_community.chat_models.tongyi import ChatTongyi
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_classic.chains import LLMChain

dotenv.load_dotenv()

chat_model =ChatTongyi(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    model_name="qwen-flash",
)

#2.提供提示词模板
prompt_template = PromptTemplate.from_template(
    template = """你可以与人类对话。

当前对话历史：{history}

人类问题：{question}

回复：
"""

)


#3.提供memory对象
memory = ConversationBufferMemory()

#4.Chain
chain = LLMChain(llm=chat_model, prompt=prompt_template, memory=memory)

#5.调用
response = chain.invoke({"question":"你好，我叫小明"})
print(response)

{'question': '你好，我叫小明', 'history': '', 'text': '你好，小明！很高兴认识你 😊 今天过得怎么样？'}


In [32]:
response = chain.invoke({"question":"我叫什么名字呢"})
print(response)

{'question': '我叫什么名字呢', 'history': 'Human: 你好，我叫小明\nAI: 你好，小明！很高兴认识你 😊 今天过得怎么样？', 'text': '你叫小明呀，记得之前你告诉过我呢！😊'}


In [ ]:
#基于例3，显示设置meory的key值
#1.获取大模型
import os
import dotenv
from langchain_community.chat_models.tongyi import ChatTongyi
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_classic.chains import LLMChain

dotenv.load_dotenv()

chat_model =ChatTongyi(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    model_name="qwen-flash",
)

#2.提供提示词模板
prompt_template = PromptTemplate.from_template(
    template = """你可以与人类对话。

当前对话历史：{chat_history}

人类问题：{question}

回复：
"""

)


#3.提供memory对象
memory = ConversationBufferMemory(memory_key="chat_history")

#4.Chain
chain = LLMChain(prompt_template=prompt_template, llm=chat_model, memory=memory)

#5.调用
response = chain.invoke({"question":"你好，我叫小明"})
print(response)

{'question': '你好，我叫小明', 'chat_history': '', 'text': '你好，小明！很高兴认识你。今天过得怎么样？'}


In [36]:
response = chain.invoke({"question":"我叫什么名字呢"})
print(response)

{'question': '我叫什么名字呢', 'chat_history': 'Human: 你好，我叫小明\nAI: 你好，小明！很高兴认识你。今天过得怎么样？', 'text': '你叫小明呀，我记得哦！'}


In [ ]:
#例5结合llm,chatprompttemplate的使用
#1.获取大模型
# 1. 导入相关包
from langchain_core.messages import SystemMessage
from langchain_classic.chains import LLMChain
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts import MessagesPlaceholder, ChatPromptTemplate, HumanMessagePromptTemplate
from langchain_community.chat_models.tongyi import ChatTongyi

# 2. 创建 LLM
llm = ChatTongyi(model_name='qwen-flash', api_key=os.getenv("DASHSCOPE_API_KEY"))

# 3. 创建 Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个与人类对话的机器人。"),
    MessagesPlaceholder(variable_name='history'),
    ("human", "问题: {question}")
])

# 4. 创建 Memory
# return_messages=True 在 ChatModel 中非常重要，它会让记忆以消息对象列表形式返回
memory = ConversationBufferMemory(memory_key="history", return_messages=True)

# 5. 创建 LLMChain (补全图片下方缺失部分)
chain = LLMChain(
    llm=llm,
    prompt=prompt,
    memory=memory
)

# 6. 调用 LLMChain
res1 = chain.invoke({"question": "中国首都在哪里？"})
print(res1, end="\n\n")

res2 = chain.invoke({"question": "我刚刚问了什么？"})
print(res2)



{'question': '中国首都在哪里？', 'history': [HumanMessage(content='中国首都在哪里？', additional_kwargs={}, response_metadata={}), AIMessage(content='中国首都是北京。', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])], 'text': '中国首都是北京。'}

{'question': '我刚刚问了什么？', 'history': [HumanMessage(content='中国首都在哪里？', additional_kwargs={}, response_metadata={}), AIMessage(content='中国首都是北京。', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='我刚刚问了什么？', additional_kwargs={}, response_metadata={}), AIMessage(content='你刚刚问的是：“中国首都在哪里？”', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])], 'text': '你刚刚问的是：“中国首都在哪里？”'}


### 3.ConversationChain的使用(存储对话)-->对ConversationBufferMemory和LLMChain进行了封装（已弃用）